In [1]:
!pip install -q timm

In [2]:
import time
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm

torch.manual_seed(42)

# Auto-detect whether this Colab TPU runtime uses torch_xla or native torch.device("tpu")
USE_XLA = False
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    USE_XLA = True
    print(f"Using PyTorch/XLA device: {device}")
except Exception:
    device = torch.device("tpu")
    print(f"Using native TorchTPU device: {device}")

def sync_tpu(tensor=None):
    if USE_XLA:
        xm.mark_step()
        xm.wait_device_ops()
    elif tensor is not None:
        _ = tensor.cpu()

class LSTMClassifier(nn.Module):
    def __init__(self, input_size=128, hidden_size=256, num_layers=2, num_classes=10):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model_builders = {
    "MobileViT-S": lambda: timm.create_model("mobilevit_s", pretrained=True).eval(),
    "MobileNetV2": lambda: timm.create_model("mobilenetv2_100", pretrained=True).eval(),
    "LSTM (2-Layer)": lambda: LSTMClassifier().eval(),
}

batch_sizes = [1, 2, 4, 8, 16, 32]
precisions = [("FP32", torch.float32), ("BF16", torch.bfloat16)]

def make_batch(model_name, batch_size, dtype):
    if model_name == "LSTM (2-Layer)":
        return torch.randn(batch_size, 128, 128, dtype=dtype)
    return torch.randn(batch_size, 3, 256, 256, dtype=dtype)

def benchmark_tpu(model, cpu_batch, batch_size, warmups=3, repeats=15):
    dev_batch = cpu_batch.to(device)
    sync_tpu(dev_batch)

    # 1. First pass: captures XLA graph compilation + first execution
    with torch.no_grad():
        t0 = time.perf_counter()
        out = model(dev_batch)
        sync_tpu(out)
        t1 = time.perf_counter()
    compile_first_ms = (t1 - t0) * 1000.0

    # 2. Additional warm-ups so caches/graphs settle
    with torch.no_grad():
        for _ in range(warmups):
            out = model(dev_batch)
            sync_tpu(out)

    # 3. Steady-State Latency (data already on TPU)
    steady_times = []
    with torch.no_grad():
        for _ in range(repeats):
            t0 = time.perf_counter()
            out = model(dev_batch)
            sync_tpu(out)
            t1 = time.perf_counter()
            steady_times.append((t1 - t0) * 1000.0)

    # 4. End-to-End Latency (H2D transfer + compute + D2H copy)
    e2e_times = []
    with torch.no_grad():
        for _ in range(repeats):
            t0 = time.perf_counter()
            x = cpu_batch.to(device)
            y = model(x)
            _ = y.cpu()
            if USE_XLA:
                xm.mark_step()
            t1 = time.perf_counter()
            e2e_times.append((t1 - t0) * 1000.0)

    med_steady = float(np.median(steady_times))
    p95_steady = float(np.percentile(steady_times, 95))
    med_e2e = float(np.median(e2e_times))
    throughput = batch_size / (med_steady / 1000.0)

    return compile_first_ms, med_steady, p95_steady, med_e2e, throughput

rows = []

for model_name, builder in model_builders.items():
    print(f"\n=== TPU Benchmarking {model_name} ===")
    for prec_name, dtype in precisions:
        model = builder().to(device=device, dtype=dtype)
        for bs in batch_sizes:
            try:
                cpu_batch = make_batch(model_name, bs, dtype)
                comp_ms, med_s, p95_s, med_e2e, tput = benchmark_tpu(model, cpu_batch, bs)
                rows.append({
                    "Model": model_name,
                    "Device": "TPU",
                    "Precision": prec_name,
                    "Batch_Size": bs,
                    "Compile_Plus_First_ms": round(comp_ms, 3),
                    "Steady_Median_ms": round(med_s, 3),
                    "Steady_P95_ms": round(p95_s, 3),
                    "E2E_Median_ms": round(med_e2e, 3),
                    "Throughput_samples_s": round(tput, 2),
                    "Status": "OK",
                })
                print(
                    f"TPU {prec_name:4} | BS={bs:2} | "
                    f"Compile+1st: {comp_ms:8.2f} ms | Steady: {med_s:7.2f} ms | "
                    f"E2E: {med_e2e:7.2f} ms | Tput: {tput:8.1f} img/s"
                )
            except RuntimeError as e:
                if "out of memory" in str(e).lower() or "resource_exhausted" in str(e).lower():
                    rows.append({
                        "Model": model_name, "Device": "TPU", "Precision": prec_name,
                        "Batch_Size": bs, "Compile_Plus_First_ms": np.nan,
                        "Steady_Median_ms": np.nan, "Steady_P95_ms": np.nan,
                        "E2E_Median_ms": np.nan, "Throughput_samples_s": np.nan,
                        "Status": "OOM",
                    })
                    print(f"TPU {prec_name:4} | BS={bs:2} | OOM!")
                else:
                    raise
        del model
        gc.collect()

tpu_df = pd.DataFrame(rows)
tpu_df.to_csv("inference_tpu.csv", index=False)
print("\nSaved inference_tpu.csv")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


Using native TorchTPU device: tpu

=== TPU Benchmarking MobileViT-S ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 22.4MB            

model.safetensors: downloading bytes:           |  0.00B            

TPU FP32 | BS= 1 | Compile+1st:  7751.24 ms | Steady:   18.11 ms | E2E:   21.63 ms | Tput:     55.2 img/s
TPU FP32 | BS= 2 | Compile+1st: 10191.67 ms | Steady:   18.41 ms | E2E:   21.27 ms | Tput:    108.6 img/s
TPU FP32 | BS= 4 | Compile+1st: 12384.58 ms | Steady:   15.22 ms | E2E:   22.85 ms | Tput:    262.9 img/s
TPU FP32 | BS= 8 | Compile+1st:  9707.33 ms | Steady:   20.58 ms | E2E:   32.40 ms | Tput:    388.7 img/s
TPU FP32 | BS=16 | Compile+1st: 12643.97 ms | Steady:   29.95 ms | E2E:   51.24 ms | Tput:    534.2 img/s
TPU FP32 | BS=32 | Compile+1st: 15325.38 ms | Steady:   54.17 ms | E2E:   99.40 ms | Tput:    590.7 img/s
TPU BF16 | BS= 1 | Compile+1st:  8748.56 ms | Steady:   18.28 ms | E2E:   19.22 ms | Tput:     54.7 img/s
TPU BF16 | BS= 2 | Compile+1st: 11303.72 ms | Steady:   16.12 ms | E2E:   15.63 ms | Tput:    124.1 img/s
TPU BF16 | BS= 4 | Compile+1st: 13714.41 ms | Steady:   24.54 ms | E2E:   25.54 ms | Tput:    163.0 img/s
TPU BF16 | BS= 8 | Compile+1st: 12157.35 ms | 

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

TPU FP32 | BS= 1 | Compile+1st:  4103.77 ms | Steady:   13.05 ms | E2E:   16.77 ms | Tput:     76.6 img/s
TPU FP32 | BS= 2 | Compile+1st:  6721.27 ms | Steady:   10.12 ms | E2E:   15.71 ms | Tput:    197.6 img/s
TPU FP32 | BS= 4 | Compile+1st:  8581.59 ms | Steady:    8.73 ms | E2E:   18.12 ms | Tput:    458.0 img/s
TPU FP32 | BS= 8 | Compile+1st:  4990.92 ms | Steady:   12.58 ms | E2E:   31.81 ms | Tput:    635.8 img/s
TPU FP32 | BS=16 | Compile+1st:  7649.60 ms | Steady:   13.59 ms | E2E:   55.51 ms | Tput:   1177.6 img/s
TPU FP32 | BS=32 | Compile+1st:  9813.43 ms | Steady:   22.72 ms | E2E:  103.61 ms | Tput:   1408.7 img/s
TPU BF16 | BS= 1 | Compile+1st:  5103.97 ms | Steady:   12.35 ms | E2E:   15.69 ms | Tput:     81.0 img/s
TPU BF16 | BS= 2 | Compile+1st:  6684.58 ms | Steady:   12.59 ms | E2E:   14.10 ms | Tput:    158.8 img/s
TPU BF16 | BS= 4 | Compile+1st:  8320.62 ms | Steady:   12.95 ms | E2E:   15.16 ms | Tput:    309.0 img/s
TPU BF16 | BS= 8 | Compile+1st:  6028.82 ms | 